<a href="https://colab.research.google.com/github/PrinzVike/CSC4110-G02/blob/main/G02_T01_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 4110 Group G2 — Task 1: Project Set-up and Dataset Acquisition
**Group:** G2 | **Module:** 3 | **Owner:** Whole group (Member 1 creates folders/repo)

Instructions to the group (Member 1 does this once on the first day of Module 3):

In Google Drive, create the shared folder CSC4110_G02 and inside it the sub-folders:

data/

notebooks/

models/

outputs/

app/

report/

tests/

Share the folder as Editor with every group member and the lecturer.

Create the GitHub repository CSC4110-G02 and add the same people as collaborators.

Every other member, once: Drive → Shared with me → right-click CSC4110_G02 → Organize → Add shortcut → My Drive so the path /content/drive/MyDrive/CSC4110_G02 works in Colab.

In [2]:
# Mount Google Drive and create (or verify) the shared group folder structure
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/CSC4110_G02"
subfolders = ["data", "notebooks", "models", "outputs", "app", "report", "tests"]

for sub in subfolders:
    path = os.path.join(BASE, sub)
    os.makedirs(path, exist_ok=True)
    print("✔ ready:", path)

print("\nProject root:", BASE)

Mounted at /content/drive
✔ ready: /content/drive/MyDrive/CSC4110_G02/data
✔ ready: /content/drive/MyDrive/CSC4110_G02/notebooks
✔ ready: /content/drive/MyDrive/CSC4110_G02/models
✔ ready: /content/drive/MyDrive/CSC4110_G02/outputs
✔ ready: /content/drive/MyDrive/CSC4110_G02/app
✔ ready: /content/drive/MyDrive/CSC4110_G02/report
✔ ready: /content/drive/MyDrive/CSC4110_G02/tests

Project root: /content/drive/MyDrive/CSC4110_G02


In [3]:
# Check that all required libraries are installed; install any missing; print each version
import importlib
import subprocess
import sys

# Map import name -> pip package name (they differ for scikit-learn)
required = {
    "numpy":      "numpy",
    "pandas":     "pandas",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
    "sklearn":    "scikit-learn",
    "joblib":     "joblib",
    "gymnasium":  "gymnasium",
    "streamlit":  "streamlit",
}

versions = {}
for import_name, pip_name in required.items():
    try:
        mod = importlib.import_module(import_name)
    except ImportError:
        print(f"{import_name}: NOT installed → installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        mod = importlib.import_module(import_name)
    versions[import_name] = getattr(mod, "__version__", "unknown")

print("\n===== Library versions =====")
for name, v in versions.items():
    print(f"{name:12s} : {v}")

streamlit: NOT installed → installing streamlit ...

===== Library versions =====
numpy        : 2.4.6
pandas       : 2.2.3
matplotlib   : 3.10.0
seaborn      : 0.13.2
sklearn      : 1.6.1
joblib       : 1.6.0
gymnasium    : 1.4.0
streamlit    : 1.67.0


In [4]:
# Download insurance.csv into the group's data folder (do not rename the file)
import urllib.request
import os

DATA_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
DATA_PATH = os.path.join(BASE, "data", "insurance.csv")

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Downloaded ->", DATA_PATH)
else:
    print("File already present ->", DATA_PATH)

print("Size (bytes):", os.path.getsize(DATA_PATH))

File already present -> /content/drive/MyDrive/CSC4110_G02/data/insurance.csv
Size (bytes): 55628


### Dataset record

| Item | Detail |
|---|---|
| **Title** | Medical Cost Personal Datasets |
| **File name (unchanged)** | `insurance.csv` |
| **Link** | https://www.kaggle.com/datasets/mirichoi0218/insurance |
| **Owner / source** | Miri Choi (Kaggle), derived from the book *Machine Learning with R* by Brett Lantz |
| **Licence** | Database: Open Database License (ODbL); Contents: Database Contents License (DbCL) v1.0 |
| **Download date** | 2025-01-01 *(replace with the actual date you ran Cell 5)* |
| **Rows × columns** | 1,338 members × 7 columns |
| **Columns** | age, sex, bmi, children, smoker, region, charges |

### Who uses the models and for what decision

Jamii Health Cover is a fictional medical insurer in Nairobi that wants premiums to reflect the real cost of each member's care and wants to spot applicants who may not have declared that they smoke. The **underwriting team** uses the regression model to set a fair yearly premium per member, while the **health-screening team** uses the classification model to decide which members to invite for a free health check. Both models therefore drive real business decisions: a wrong charge estimate over- or under-prices a member, and a missed smoker means the insurer charges too little for a high-cost risk.

### Targets

| Model | Target |
|---|---|
| Regression | `charges` — yearly medical cost in US dollars |
| Classification | `smoker` — yes / no |

### Expected best predictors (with reasons)

- **`smoker`** — smokers in this dataset pay on average about $32,000 per year vs about $8,400 for non-smokers, so it is the dominant driver of `charges` and the target of the classifier.
- **`bmi`** — obesity drives chronic disease and cost; the effect is much stronger for smokers (the `bmi × smoker` interaction).
- **`age`** — medical cost rises steadily with age.
- **`children`** — larger families can incur slightly higher yearly charges.
- **`region` / `sex`** — weaker predictors, but they are kept as fairness-check columns.
- **`charges`** — for the *classifier*, `charges` is a very strong predictor of `smoker` because the two cost distributions barely overlap.

In [5]:
# Import insurance.csv with pandas, show first five rows and the dataset shape
import pandas as pd

df = pd.read_csv(DATA_PATH)
print("First five rows:")
display(df.head())
print("\nShape (rows, columns):", df.shape)
print("\nColumns:", list(df.columns))

First five rows:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Shape (rows, columns): (1338, 7)

Columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']


### Summary
- Shared Drive folder **CSC4110_G02** and GitHub repo **CSC4110-G02** created (Part a).
- All required libraries verified/installed with versions printed (Part b).
- `insurance.csv` downloaded unchanged into `data/` and documented (Part c).
- Users, decisions, targets and expected predictors documented (Part d).
- Dataset loaded successfully: **1,338 rows × 7 columns** (Part e).

**Graded evidence:** variables `df` and `BASE`; file `data/insurance.csv`.
**Save as:** `notebooks/G02_T01_setup.ipynb`.